# LifeMate: Kaggle GPU training and evaluation

This notebook uses attached local assets only. It discovers data/model mount points, validates the official train labels, runs a base-model smoke test, and runs optional training/evaluation stages when their flags are enabled. No hosted LLM API is used. Kaggle execution is gated by the attached inputs and available GPU.

In [ ]:
# 0. Reproducible run switches and persistent output root
from pathlib import Path
import os, json, sys, shutil, subprocess, time, zipfile

RUN_ROOT = Path('/kaggle/working/lifemate_run')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
# Keep full training and public inference opt-in so a top-to-bottom smoke run is bounded.
INSTALL_STACK = False  # only enable after reviewing requirements/kaggle.txt and Kaggle CUDA
RUN_MODEL_SMOKE = False
RUN_BASELINE_FULL = False
RUN_FULL_TRAINING = False
RUN_FINETUNED_FULL = False
RUN_AGENT_LOOP_FULL = False
RUN_PUBLIC_TEST = False
MAX_SMOKE_SAMPLES = 5
MAX_LOOP_SMOKE_SAMPLES = 2
SEED = 42
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'


## 1. Code setup
Attach this repository as a Kaggle Dataset, or upload a ZIP containing only source/config/docs. Optionally set `LIFEMATE_GIT_URL` for a one-time code clone when notebook internet is enabled. The dataset/model files remain separate Kaggle inputs.

In [ ]:
# Locate project code from an input directory; ZIP fallback is for source code only.
INPUT_ROOT = Path('/kaggle/input')
project_candidates = [p.parent for p in INPUT_ROOT.rglob('pyproject.toml') if (p.parent/'configs/paths.yaml').is_file()] if INPUT_ROOT.exists() else []
if project_candidates:
    PROJECT = project_candidates[0].resolve()
else:
    zip_candidates = list(INPUT_ROOT.rglob('*.zip')) if INPUT_ROOT.exists() else []
    PROJECT = None
    for zpath in zip_candidates:
        dest = Path('/kaggle/working/lifemate_source')
        with zipfile.ZipFile(zpath) as zf:
            names = zf.namelist()
            if not any(n.endswith('pyproject.toml') for n in names):
                continue
            if any(Path(n).is_absolute() or '..' in Path(n).parts for n in names):
                raise ValueError(f'Unsafe source ZIP paths: {zpath}')
            zf.extractall(dest)
        found = list(dest.rglob('pyproject.toml'))
        if found:
            PROJECT = found[0].parent
            break
if PROJECT is None and os.environ.get('LIFEMATE_GIT_URL'):
    PROJECT = Path('/kaggle/working/lifemate_repo')
    subprocess.run(['git','clone','--depth','1',os.environ['LIFEMATE_GIT_URL'],str(PROJECT)],check=True)
if PROJECT is None:
    raise FileNotFoundError('Attach project source as Kaggle Dataset/ZIP or set LIFEMATE_GIT_URL for code transfer only.')
sys.path.insert(0, str(PROJECT/'src'))
print('Project:', PROJECT)


## 2. Inspect runtime and discover attached inputs
No Kaggle device is assumed. The discovery checks file markers and model identity rather than relying on dataset slugs.

In [ ]:
from lifemate.utils.hardware import inspect_hardware
hardware = inspect_hardware(['/kaggle/input','/kaggle/working'])
print(json.dumps(hardware, indent=2, default=str))
if not hardware['cuda_available']:
    print('No CUDA GPU: data checks can run, but QLoRA/model smoke stages will be skipped.')


In [ ]:
# Discover SFT/original/model/ASR mounts independently.
all_files = list(INPUT_ROOT.rglob('*')) if INPUT_ROOT.exists() else []
sft_candidates = sorted({p.parent for p in all_files if p.name == 'train_all_lengths.jsonl' and (p.parent/'split_manifest.json').is_file()})
data_candidates = sorted({p.parent.parent for p in all_files if p.name == 'train.jsonl' and p.parent.name == 'train' and (p.parent.parent/'public_test/public_test.jsonl').is_file() and (p.parent/'zaic_tools').is_dir()})
model_candidates=[]; asr_candidates=[]
for cfg_path in (p for p in all_files if p.name == 'config.json'):
    try: cfg=json.loads(cfg_path.read_text(encoding='utf-8'))
    except Exception: continue
    model_id=(cfg.get('_name_or_path') or '').lower()
    source_path=cfg_path.parent/'model_source.json'
    if source_path.exists():
        try: model_id=json.loads(source_path.read_text(encoding='utf-8')).get('model_id','').lower()
        except Exception: pass
    if cfg.get('model_type') == 'qwen3' and 'qwen3-4b-instruct-2507' in model_id:
        model_candidates.append(cfg_path.parent)
    if cfg.get('model_type') == 'whisper' and (cfg_path.parent/'preprocessor_config.json').exists():
        asr_candidates.append(cfg_path.parent)
if not sft_candidates or not data_candidates or not model_candidates:
    raise FileNotFoundError(json.dumps({'sft_candidates':list(map(str,sft_candidates)),'original_candidates':list(map(str,data_candidates)),'qwen_candidates':list(map(str,model_candidates))},indent=2))
SFT_DIR=sft_candidates[0]; ORIGINAL_DIR=data_candidates[0]; MODEL_DIR=model_candidates[0]
ASR_DIR=Path(os.environ['ASR_MODEL_DIR']) if os.environ.get('ASR_MODEL_DIR') else (asr_candidates[0] if asr_candidates else None)
os.environ.update({'SFT_DATA_DIR':str(SFT_DIR),'ORIGINAL_DATA_DIR':str(ORIGINAL_DIR),'BASE_MODEL_DIR':str(MODEL_DIR),'LIFEMATE_WORK_DIR':str(RUN_ROOT)})
if ASR_DIR: os.environ['ASR_MODEL_DIR']=str(ASR_DIR)
print(json.dumps({'SFT_DIR':str(SFT_DIR),'ORIGINAL_DIR':str(ORIGINAL_DIR),'MODEL_DIR':str(MODEL_DIR),'ASR_DIR':str(ASR_DIR) if ASR_DIR else None},indent=2))


## 3. Dependency verification
The notebook does not upgrade a preinstalled CUDA/PyTorch stack implicitly. The lock file is a candidate environment; inspect the printed versions and install it only through an explicitly reviewed Kaggle session.

In [ ]:
import importlib.metadata as im
versions={name:(im.version(name) if __import__('importlib').util.find_spec(name.replace('-','_')) else None) for name in ['torch','transformers','datasets','accelerate','peft','bitsandbytes','trl','PyYAML']}
print(json.dumps(versions,indent=2))
if INSTALL_STACK:
    subprocess.run([sys.executable,'-m','pip','install','-r',str(PROJECT/'requirements/kaggle.txt')],check=True)
    raise RuntimeError('Dependencies installed. Restart the Kaggle session, then rerun from the top so CUDA libraries are loaded cleanly.')
import torch, transformers
if tuple(int(x) for x in transformers.__version__.split('.')[:2]) < (4,51): raise RuntimeError('Qwen3 support needs transformers>=4.51.0; review requirements/kaggle.txt')
if tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) < (2,6): raise RuntimeError('Qwen3 recommended stack requires torch>=2.6; inspect Kaggle CUDA before changing it')


## 4. Validate data and official ToolEnv
This stage reads labeled train and the public input metadata. Only train samples are replayed; public test is not used for training.

In [ ]:
def run_script(name, *args):
    cmd=[sys.executable,str(PROJECT/'scripts'/name),*map(str,args)]
    print('>', ' '.join(cmd))
    subprocess.run(cmd,cwd=PROJECT,check=True)
run_script('audit_data.py','--sft-dir',SFT_DIR,'--original-dir',ORIGINAL_DIR,'--output-dir',RUN_ROOT/'audit')
run_script('prepare_data.py','--sft-dir',SFT_DIR,'--original-dir',ORIGINAL_DIR,'--output-dir',RUN_ROOT/'processed')
run_script('verify_toolenv.py','--sft-dir',SFT_DIR,'--original-dir',ORIGINAL_DIR,'--output',RUN_ROOT/'audit/toolenv_verification.json')


## 5. Verify exact model, tokenizer, template, context, and GPU fit
The selected variant is the non-thinking `Qwen3-4B-Instruct-2507`; it uses its shipped chat template. The model card states 262,144 native context, but training length is selected conservatively from GPU VRAM and measured tokenized examples. No thinking-mode flag is injected.

In [ ]:
from lifemate.models.loader import inspect_local_model
model_info=inspect_local_model(MODEL_DIR)
print(json.dumps(model_info,indent=2,default=str))
if model_info.get('model_type')!='qwen3' or model_info.get('model_id_verified')!='Qwen/Qwen3-4B-Instruct-2507': raise RuntimeError('The attached base model is not the requested exact Qwen3-4B-Instruct-2507.')
from transformers import AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained(str(MODEL_DIR),local_files_only=True,trust_remote_code=False)
assert tokenizer.chat_template, 'Tokenizer chat template missing'
print('EOS:',tokenizer.eos_token,tokenizer.eos_token_id,'PAD:',tokenizer.pad_token,tokenizer.pad_token_id)
print('Chat template prefix:',tokenizer.chat_template[:500])


## 6. Model load and forward/generation smoke test
Run only when the dynamically detected GPU has enough free VRAM. The script loads local assets with `local_files_only=True` and records a generated sample.

In [ ]:
FREE_GIB=(hardware['gpu'][0]['free_vram_bytes']/1024**3) if hardware['gpu'] else 0
if RUN_MODEL_SMOKE:
    if FREE_GIB < 6: raise RuntimeError(f'Only {FREE_GIB:.1f} GiB VRAM free; stopping before model load.')
    run_script('model_smoke.py','--model-dir',MODEL_DIR,'--output',RUN_ROOT/'model_smoke.json')
else: print('Model forward smoke skipped by RUN_MODEL_SMOKE=False; no model inference claimed.')


## 7. Baseline evaluation
Baseline is always the unmodified local base model. Smoke evaluation and full validation are separate.

In [ ]:
if RUN_BASELINE_FULL:
    run_script('evaluate_baseline.py','--model-dir',MODEL_DIR,'--original-dir',ORIGINAL_DIR,'--sft-dir',SFT_DIR,'--data',RUN_ROOT/'processed/validation.jsonl','--output-dir',RUN_ROOT/'baseline')
else:
    print('Full baseline evaluation skipped. Enable RUN_BASELINE_FULL after a successful model smoke and runtime estimate.')


## 8. QLoRA smoke step and training
Training is opt-in. `scripts/train.py` checks Linux/CUDA/VRAM, exact local model identity, detects LoRA modules by inspecting the loaded architecture, runs one reversible forward/backward update, records sequence exclusions, then trains and saves resumable checkpoints plus the best adapter.

In [ ]:
adapter_dir=RUN_ROOT/'lora/run1/adapter'
if RUN_FULL_TRAINING:
    if not hardware['cuda_available']: raise RuntimeError('CUDA not available; QLoRA stopped.')
    run_script('train.py','--model-dir',MODEL_DIR,'--train-jsonl',RUN_ROOT/'processed/train.jsonl','--validation-jsonl',RUN_ROOT/'processed/validation.jsonl','--output-dir',RUN_ROOT/'lora/run1')
    run_script('select_checkpoint.py','--model-dir',MODEL_DIR,'--checkpoint-root',RUN_ROOT/'lora/run1/checkpoints','--original-dir',ORIGINAL_DIR,'--sft-dir',SFT_DIR,'--output-dir',RUN_ROOT/'lora/run1/checkpoint_selection')
    adapter_dir=RUN_ROOT/'lora/run1/checkpoint_selection/adapter_selected'
else: print('QLoRA not run; set RUN_FULL_TRAINING=True after reviewing GPU and cost estimate.')


## 9. Fine-tuned evaluation and comparison

In [ ]:
if RUN_FINETUNED_FULL:
    if not (adapter_dir/'adapter_config.json').exists(): raise FileNotFoundError('No adapter checkpoint found; training must finish first or attach an exported adapter.')
    run_script('evaluate_finetuned.py','--model-dir',MODEL_DIR,'--adapter-dir',adapter_dir,'--original-dir',ORIGINAL_DIR,'--sft-dir',SFT_DIR,'--data',RUN_ROOT/'processed/validation.jsonl','--output-dir',RUN_ROOT/'finetuned')
    run_script('compare_models.py','--baseline-steps',RUN_ROOT/'baseline/steps/metrics.json','--finetuned-steps',RUN_ROOT/'finetuned/steps/metrics.json','--baseline-trajectories',RUN_ROOT/'baseline/trajectories/trajectory_evaluation.json','--finetuned-trajectories',RUN_ROOT/'finetuned/trajectories/trajectory_evaluation.json','--output',RUN_ROOT/'evaluation/model_comparison.json')
else: print('Fine-tuned evaluation/comparison skipped.')


## 10. Full Agent Loop evaluation
The trajectory evaluator runs one official ToolEnv per labeled validation sample and injects verified observations between decisions. Ground truth is used only for evaluation, never as policy input.

In [ ]:
if RUN_AGENT_LOOP_FULL:
    loop_adapter_args=['--adapter-dir',str(adapter_dir)] if (adapter_dir/'adapter_config.json').exists() else []
    run_script('evaluate_trajectories.py','--original-dir',ORIGINAL_DIR,'--sft-dir',SFT_DIR,'--model-dir',MODEL_DIR,*loop_adapter_args,'--output-dir',RUN_ROOT/'agent_loop')
else: print('Full Agent Loop evaluation skipped.')


## 11. Public test inference
Public audio requires an attached offline Whisper-family ASR checkpoint in addition to Qwen. Public test has no reference labels; inference counts are not accuracy scores.

In [ ]:
if RUN_PUBLIC_TEST:
    if ASR_DIR is None: raise FileNotFoundError('Attach a local ASR model and set/discover ASR_MODEL_DIR; no hosted transcription is used.')
    run_script('run_public_test.py','--original-dir',ORIGINAL_DIR,'--model-dir',MODEL_DIR,'--adapter-dir',adapter_dir if (adapter_dir/'adapter_config.json').exists() else '', '--asr-model-dir',ASR_DIR,'--output',RUN_ROOT/'submission/submission.jsonl')
    run_script('build_submission.py','--original-dir',ORIGINAL_DIR,'--submission',RUN_ROOT/'submission/submission.jsonl')
else: print('Public inference skipped.')


## 12. Offline deployment checks and artifact export
The adapter does not replace the base model; both base and adapter, plus offline ASR for audio inputs, are required for the full audio submission pipeline.

In [ ]:
adapter_arg=['--adapter-dir',str(adapter_dir)] if (adapter_dir/'adapter_config.json').exists() else []
if model_info and (RUN_MODEL_SMOKE or RUN_FULL_TRAINING):
    asr_arg=['--asr-model-dir',str(ASR_DIR)] if ASR_DIR else []
    run_script('offline_check.py','--model-dir',MODEL_DIR,*adapter_arg,*asr_arg,'--output',RUN_ROOT/'offline_check.json')
# Archive code/report/config/metrics/adapter and submission only; never copy attached datasets or base weights.
export=Path('/kaggle/working/lifemate_export'); export.mkdir(parents=True,exist_ok=True)
shutil.copytree(PROJECT/'reports',export/'reports',dirs_exist_ok=True)
shutil.copytree(RUN_ROOT,export/'run_artifacts',dirs_exist_ok=True)
archive=shutil.make_archive('/kaggle/working/lifemate_run_export','zip',root_dir=export)
print('Export:',archive)
